In [1]:
import numpy as np
import pandas as pd
import torch
import random

from Conv1DAE import Conv1DAE, detect_anomalies_conv1dae, conv1dae_train
from optimize_models import latency_to_detection, get_basic_metrics, train_test_split_anomaly_sequence, optimize_conv1dae
from prepare_data import load_or_cache_twitter, load_or_cache_mit_bih, load_or_cache_bonn, \
    get_mit_bih_segments, get_twitter_segments, get_bonn_segments
from visualizations import heatmaps, segments_reconstruction

torch.manual_seed(0)
random.seed(0)

#### <center>Zbiór EEG Bonn</center>

Uznajemy, że zbiór E to outliery, a reszta:
- zdrowi oczy otwarte -> A,
- zdrowi oczy zamknięte -> B,
- pacjenci między napadami (zdrowa półkula) -> C,
- pacjenci między napadami (strefa padaczkowa) -> D,

są zdrowi.

In [2]:
eeg_bonn_dataset = load_or_cache_bonn()

Przygotowany zbiór EEG-Bonn

Każda sekwencja ma przypisaną etykietę na podstawie przynależności do zbioru. Etykieta segmentu jest przypisana na podstawie stosunku liczby anomalii do normalnych próbek na poziomie sekwencji.

In [3]:
bonn_overlaps = [0.25, 0.5, 0.75]
bonn_window_sizes = [2, 3, 5]

<center>Eksperymenty dla 1D Conv-AE</center>

In [4]:
conv1dae_bonn_experiments, conv1dae_bonn_heatmap, conv1dae_bonn_training_params = [
    {
        latent: {
            s: {
                o: None for o in bonn_overlaps
            } for s in bonn_window_sizes
        } for latent in [8, 16]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_bonn_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            # Przygotowanie danych
            X, y, _ = get_bonn_segments(eeg_bonn_dataset, second, overlap)

            # Podział na train/test
            X_train, X_test, y_train, y_test = train_test_split_anomaly_sequence(X, y, random_state=42)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_study = optimize_conv1dae(
                X=X_train,
                y=y_train,
                latent=latent,
                dataset_name="EEG Bonn",
                n_trials=10
            )
            conv1dae_params = conv1dae_study.best_params
            conv1dae_train_params = {k: conv1dae_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_loss_params = {k: conv1dae_params[k] for k in ["alpha", "beta", "gamma"]}

            conv1dae_bonn_training_params[latent][second][overlap] = conv1dae_params

            # Konwersja z numpy do torch
            X_train, X_test = torch.from_numpy(X_train).to(dtype=torch.float32, device='cuda'), torch.from_numpy(X_test).to(dtype=torch.float32, device='cuda')

            # Autoenkoder
            conv1dae = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train = X_train[:, :, None, :]
            X_test = X_test[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae,
                data=X_train,
                **conv1dae_train_params
            )

            # Wyznaczenie progu z danych treningowych
            y_train_scores, reconstruction, encoded = detect_anomalies_conv1dae(
                model=conv1dae,
                data=X_train,
                **conv1dae_loss_params
            )
            train_threshold = np.percentile(y_train_scores, 99)

            # Wyznaczenie etykiet -1/1 na zbiorze testowym
            y_test_scores, reconstruction, encoded = detect_anomalies_conv1dae(
                model=conv1dae,
                data=X_test,
                **conv1dae_loss_params
            )
            y_pred = np.where(y_test_scores > train_threshold, -1, 1)

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_true=y_test.reshape(-1, 1).squeeze(),
                y_pred=y_pred.reshape(-1, 1).squeeze(),
                y_scores=y_test_scores.reshape(-1, 1).squeeze()
            )
            conv1dae_test_latencies = latency_to_detection(y_test, y_pred)
            conv1dae_bonn_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_test_latencies}
            conv1dae_bonn_heatmap[latent][second][overlap] = y_test_scores

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test.cpu().numpy().squeeze(2),
                X_pred=reconstruction,
                y_true=y_test,
                y_pred=y_pred.squeeze(),
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_bonn.png"
            )

[I 2026-01-25 14:44:51,792] A new study created in memory with name: Optuna for Conv1DAE on EEG Bonn dataset


Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3


In [5]:
conv1dae_bonn_experiments

{8: {2: {0.25: {'pr_auc': 0.9807525500289108,
    'f1': 0.8275593616193071,
    'recall': 0.7086666666666667,
    'detection_rate': 0.92,
    'detected_anomalies': 92,
    'missed_anomalies': 8,
    'total_anomalies': 100,
    'mean_latency': np.float64(0.9347826086956522),
    'median_latency': np.float64(0.0)},
   0.5: {'pr_auc': 0.9858869544704055,
    'f1': 0.7835676867934932,
    'recall': 0.6459090909090909,
    'detection_rate': 0.84,
    'detected_anomalies': 84,
    'missed_anomalies': 16,
    'total_anomalies': 100,
    'mean_latency': np.float64(1.6547619047619047),
    'median_latency': np.float64(0.0)},
   0.75: {'pr_auc': 0.9839140451548168,
    'f1': 0.8376427352670954,
    'recall': 0.7252272727272727,
    'detection_rate': 0.95,
    'detected_anomalies': 95,
    'missed_anomalies': 5,
    'total_anomalies': 100,
    'mean_latency': np.float64(2.536842105263158),
    'median_latency': np.float64(0.0)}},
  3: {0.25: {'pr_auc': 0.9865749810460362,
    'f1': 0.849770642201

In [6]:
conv1dae_bonn_training_params

{8: {2: {0.25: {'alpha': 1.5102922471479012,
    'beta': 0.39097295564859835,
    'gamma': 1.416221320400211,
    'lr': 0.0026587543983272706,
    'epochs': 29},
   0.5: {'alpha': 1.5917022549267168,
    'beta': 0.19127267288786132,
    'gamma': 1.262378215816119,
    'lr': 0.007309539835912915,
    'epochs': 32},
   0.75: {'alpha': 1.5102922471479012,
    'beta': 0.39097295564859835,
    'gamma': 1.416221320400211,
    'lr': 0.0026587543983272706,
    'epochs': 29}},
  3: {0.25: {'alpha': 1.5102922471479012,
    'beta': 0.39097295564859835,
    'gamma': 1.416221320400211,
    'lr': 0.0026587543983272706,
    'epochs': 29},
   0.5: {'alpha': 1.5102922471479012,
    'beta': 0.39097295564859835,
    'gamma': 1.416221320400211,
    'lr': 0.0026587543983272706,
    'epochs': 29},
   0.75: {'alpha': 1.5102922471479012,
    'beta': 0.39097295564859835,
    'gamma': 1.416221320400211,
    'lr': 0.0026587543983272706,
    'epochs': 29}},
  5: {0.25: {'alpha': 1.5917022549267168,
    'beta': 0.

In [7]:
for latent in conv1dae_bonn_heatmap.keys():
    heatmaps(conv1dae_bonn_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze EEG Bonn", f"{latent}_conv1dae_heatmap_bonn")

#### <center>Zbiór MIT-BIH ECG</center>

Dzięki plikom atr mam dostęp, w którym momencie zostało zarejestrowane uderzenie serca. Dzięki temu mogę każdy z segmentów w sekwencjach oznaczać, ale może być wiele etykiet w segmencie. Aby przypisać czy jest outlierem zliczam wystąpienia "N" i pozostałych i porównuje, czego jest więcej.

In [8]:
mit_bih = load_or_cache_mit_bih()

Przygotowany zbiór MIT BIH

Każdy segment posiada etykietę przypisaną na podstawie stosunku liczby uderzeń normalnych do arytmii.

In [9]:
mit_bih_overlaps = [0.25, 0.5, 0.75]
mit_bih_window_sizes = [2, 3, 5]

<center>Eksperymenty dla 1D Conv-AE</center>

In [10]:
conv1dae_mit_bih_experiments, conv1dae_mit_bih_heatmap, conv1dae_mit_bih_training_params = [
    {
        latent: {
            s: {
                o: None for o in mit_bih_overlaps
            } for s in mit_bih_window_sizes
        } for latent in [8, 16]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_mit_bih_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            torch.cuda.empty_cache()
            X, y, _ = get_mit_bih_segments(mit_bih, second, overlap)

            # Wybieramy te anomalie, które mają najmniej zanieczyszczonych segmentów
            anomaly_ratios = (y == -1).mean(axis=1)
            pseudo_y = np.where(anomaly_ratios < 0.3, 1, -1)

            # Podział na train/test
            X_train, X_test, y_train, y_test = train_test_split_anomaly_sequence(X, y, pseudo_label=pseudo_y)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_study = optimize_conv1dae(
                X=X_train,
                y=y_train,
                latent=latent,
                dataset_name="MIT BIH",
                n_trials=20,
                percentile=99,
                show_optuna_output=True
            )

            conv1dae_params = conv1dae_study.best_params
            conv1dae_train_params = {k: conv1dae_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_loss_params = {k: conv1dae_params[k] for k in ["alpha", "beta", "gamma"]}

            conv1dae_mit_bih_training_params[latent][second][overlap] = conv1dae_params

            # Konwersja z numpy do torch
            X_train, X_test = torch.from_numpy(X_train).to(dtype=torch.float32, device="cuda"), torch.from_numpy(X_test).to(dtype=torch.float32, device="cuda")

            # Autoenkoder
            conv1dae = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train = X_train[:, :, None, :]
            X_test = X_test[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae,
                data=X_train,
                **conv1dae_train_params
            )

            # Wyznaczenie progu z danych treningowych
            y_train_scores, reconstruction, encoded = detect_anomalies_conv1dae(
                model=conv1dae,
                data=X_train,
                **conv1dae_loss_params
            )
            train_threshold = np.percentile(y_train_scores, 99)

            # Wyznaczenie etykiet -1/1 na zbiorze testowym
            y_test_scores, reconstruction, encoded = detect_anomalies_conv1dae(
                model=conv1dae,
                data=X_test,
                **conv1dae_loss_params
            )
            y_pred = np.where(y_test_scores > train_threshold, -1, 1)

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_test.reshape(-1, 1).squeeze(),
                y_pred.reshape(-1, 1).squeeze(),
                y_test_scores.reshape(-1, 1).squeeze()
            )
            conv1dae_mit_bih_test_latencies = latency_to_detection(y_test, y_pred)
            conv1dae_mit_bih_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_mit_bih_test_latencies}
            conv1dae_mit_bih_heatmap[latent][second][overlap] = y_test_scores

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test.cpu().numpy().squeeze(2),
                X_pred=reconstruction,
                y_true=y_test,
                y_pred=y_pred.squeeze(),
                n=2,
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_mit_bih.png"
            )

Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3


In [11]:
conv1dae_mit_bih_experiments

{8: {2: {0.25: {'pr_auc': 0.4401703469342789,
    'f1': 0.038846737481031866,
    'recall': 0.020004167534903104,
    'detection_rate': 0.03768318213538032,
    'detected_anomalies': 54,
    'missed_anomalies': 1379,
    'total_anomalies': 1433,
    'mean_latency': np.float64(7.555555555555555),
    'median_latency': np.float64(1.0)},
   0.5: {'pr_auc': 0.4134114669147881,
    'f1': 0.031993492848912085,
    'recall': 0.016411682892906815,
    'detection_rate': 0.04098820887142055,
    'detected_anomalies': 73,
    'missed_anomalies': 1708,
    'total_anomalies': 1781,
    'mean_latency': np.float64(7.219178082191781),
    'median_latency': np.float64(1.0)},
   0.75: {'pr_auc': 0.42542344297523516,
    'f1': 0.036506655909640987,
    'recall': 0.018860715526224384,
    'detection_rate': 0.059024390243902436,
    'detected_anomalies': 121,
    'missed_anomalies': 1929,
    'total_anomalies': 2050,
    'mean_latency': np.float64(17.867768595041323),
    'median_latency': np.float64(2.0)}

In [12]:
conv1dae_mit_bih_training_params

{8: {2: {0.25: {'alpha': 1.5779972601681014,
    'beta': 0.11742508365045984,
    'gamma': 1.4330880728874675,
    'lr': 0.015930522616241012,
    'epochs': 43},
   0.5: {'alpha': 1.6321961521652106,
    'beta': 0.17192971242569716,
    'gamma': 1.2976749945804005,
    'lr': 0.007311400508855897,
    'epochs': 44},
   0.75: {'alpha': 1.719040463730915,
    'beta': 0.10288527735688902,
    'gamma': 1.1334128008354283,
    'lr': 0.00467294158897531,
    'epochs': 41}},
  3: {0.25: {'alpha': 1.6750053868650032,
    'beta': 0.10442602151007785,
    'gamma': 1.1657297222764424,
    'lr': 0.0041158805795231896,
    'epochs': 34},
   0.5: {'alpha': 1.831261142176991,
    'beta': 0.19351332282682332,
    'gamma': 1.2600340105889054,
    'lr': 0.0123999678368461,
    'epochs': 29},
   0.75: {'alpha': 1.6334889314761196,
    'beta': 0.10394209471534252,
    'gamma': 1.3050789532666531,
    'lr': 0.035443165319224056,
    'epochs': 50}},
  5: {0.25: {'alpha': 1.5779972601681014,
    'beta': 0.117

In [13]:
for latent in conv1dae_mit_bih_heatmap.keys():
    heatmaps(conv1dae_mit_bih_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze MIT BIH", f"{latent}_conv1dae_heatmap_mit_bih")

#### <center>Zbiór Numenta Anomaly Benchmark (Twitter)</center>

In [14]:
twitter = load_or_cache_twitter()

Przygotowany zbiór Twitter

Każdy segment posiada etykietę przypisaną na podstawie stosunku liczby anomalii do liczby prawidłowych próbek.

In [15]:
twitter_window_sizes = [15 * 12, 20 * 12, 30 * 12]
twitter_overlaps = [0.25, 0.5, 0.75]

<center>Eksperymenty dla 1D Conv-AE</center>

In [16]:
conv1dae_twitter_experiments, conv1dae_twitter_heatmap, conv1dae_twitter_training_params = [
    {
        latent: {
            s: {
                o: None for o in twitter_overlaps
            } for s in twitter_window_sizes
        } for latent in [8, 16]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_twitter_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            # Przygotowanie danych
            X, y, _ = get_twitter_segments(twitter, second, overlap)

            # Podział na train/test
            X_train, X_test, y_train, y_test = train_test_split_anomaly_sequence(X, y, random_state=42)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_study = optimize_conv1dae(
                X=X_train,
                y=y_train,
                latent=latent,
                dataset_name="Twitter",
                n_trials=100,
                percentile=99
            )
            conv1dae_params = conv1dae_study.best_params
            conv1dae_train_params = {k: conv1dae_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_loss_params = {k: conv1dae_params[k] for k in ["alpha", "beta", "gamma"]}
            conv1dae_twitter_training_params[latent][second][overlap] = conv1dae_params

            # Konwersja z numpy do torch
            X_train, X_test = torch.from_numpy(X_train).to(dtype=torch.float32, device="cuda"), torch.from_numpy(X_test).to(dtype=torch.float32, device="cuda")

            # Autoenkoder
            conv1dae = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train = X_train[:, :, None, :]
            X_test = X_test[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae,
                data=X_train,
                **conv1dae_train_params
            )

            # Wyznaczenie progu z danych treningowych
            y_train_scores, reconstruction, encoded = detect_anomalies_conv1dae(
                model=conv1dae,
                data=X_train,
                **conv1dae_loss_params
            )
            train_threshold = np.percentile(y_train_scores, 99)

            # Wyznaczenie etykiet -1/1 na zbiorze testowym
            y_test_scores, reconstruction, encoded = detect_anomalies_conv1dae(
                model=conv1dae,
                data=X_test,
                **conv1dae_loss_params
            )
            y_pred = np.where(y_test_scores > train_threshold, -1, 1)

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_test.reshape(-1, 1).squeeze(),
                y_pred.reshape(-1, 1).squeeze(),
                y_test_scores.reshape(-1, 1).squeeze()
            )
            conv1dae_test_latencies = latency_to_detection(y_test, y_pred)
            conv1dae_twitter_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_test_latencies}
            conv1dae_twitter_heatmap[latent][second][overlap] = y_test_scores

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test.cpu().numpy().squeeze(2),
                X_pred=reconstruction,
                y_true=y_test,
                y_pred=y_pred.squeeze(),
                n=2,
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_twitter.png"
            )

Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454


In [17]:
conv1dae_twitter_experiments

{8: {180: {0.25: {'pr_auc': 0.045638991966371716,
    'f1': 0.06134969325153374,
    'recall': 0.08333333333333333,
    'detection_rate': 0.25,
    'detected_anomalies': 1,
    'missed_anomalies': 3,
    'total_anomalies': 4,
    'mean_latency': np.float64(0.0),
    'median_latency': np.float64(0.0)},
   0.5: {'pr_auc': 0.04543130957859833,
    'f1': 0.058577405857740586,
    'recall': 0.07692307692307693,
    'detection_rate': 0.25,
    'detected_anomalies': 1,
    'missed_anomalies': 3,
    'total_anomalies': 4,
    'mean_latency': np.float64(0.0),
    'median_latency': np.float64(0.0)},
   0.75: {'pr_auc': 0.04456231460406523,
    'f1': 0.06611570247933884,
    'recall': 0.08888888888888889,
    'detection_rate': 0.25,
    'detected_anomalies': 1,
    'missed_anomalies': 3,
    'total_anomalies': 4,
    'mean_latency': np.float64(0.0),
    'median_latency': np.float64(0.0)}},
  240: {0.25: {'pr_auc': 0.04528462012116981,
    'f1': 0.06451612903225806,
    'recall': 0.090909090909090

In [18]:
conv1dae_twitter_training_params

{8: {180: {0.25: {'alpha': 1.509310947876131,
    'beta': 0.1940788293534546,
    'gamma': 1.394827730887482,
    'lr': 0.03076948959423434,
    'epochs': 35},
   0.5: {'alpha': 1.5869680700688058,
    'beta': 0.14972367514340232,
    'gamma': 1.0255690107395545,
    'lr': 0.03251204692926416,
    'epochs': 42},
   0.75: {'alpha': 1.5519929901128067,
    'beta': 0.17899288014113388,
    'gamma': 1.1790657254719343,
    'lr': 0.038829341463123994,
    'epochs': 45}},
  240: {0.25: {'alpha': 1.5199815787611834,
    'beta': 0.18728910039158422,
    'gamma': 1.2174411215081558,
    'lr': 0.03244188929174148,
    'epochs': 37},
   0.5: {'alpha': 1.5865585220715954,
    'beta': 0.33325639224853554,
    'gamma': 1.1123018319808726,
    'lr': 0.019584332496973934,
    'epochs': 45},
   0.75: {'alpha': 1.5069774849103532,
    'beta': 0.23520405524239896,
    'gamma': 1.0702037129147761,
    'lr': 0.05836169590677858,
    'epochs': 45}},
  360: {0.25: {'alpha': 1.5783623896391834,
    'beta': 0.

Heatmapy

In [19]:
for latent in conv1dae_twitter_heatmap.keys():
    heatmaps(conv1dae_twitter_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze Twitter", f"{latent}_conv1dae_heatmap_twitter")